# Checkpoint 1: Tumor Surface Target Prioritization

#### Disclaimer: ChatGPT was used to help guide the layout of the notebook, provide useful libraries to reach our goal, and provide guidance at roadblocks.

## Part 1: Locate LUAD Gene Expression file and load the dataset

In [1]:
import pandas as pd
from pathlib import Path

folder = Path.home() / "Desktop" / "Fall 2026 Classes"

for file in folder.rglob("*"):
    if "HiSeqV2" in file.name:
        print(file)


/Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/TCGA.LUAD.sampleMap_HiSeqV2.gz


In [2]:
files = list(folder.rglob("*HiSeqV2*"))

file_path = files[0]

expression = pd.read_csv(
    file_path,
    sep="\t",
    compression="infer",
    index_col=0
)

print("Dataset shape:", expression.shape)
expression.iloc[:5, :5]

Dataset shape: (20530, 576)


,TCGA-69-7978-01,TCGA-62-8399-01,TCGA-78-7539-01,TCGA-50-5931-11,TCGA-73-4658-01
sample,,,,,
ARHGEF10L,9.9898,10.4257,9.6264,8.6835,9.2078
HIF3A,4.2598,11.6239,9.1362,9.4824,5.0288
RNF17,0.4181,0.0000,1.1231,0.8221,0.0000
RNF10,10.3657,11.5489,11.6692,11.7341,11.6209
RNF11,11.1718,11.0200,10.4679,11.6787,11.3414


## Part 2: Calculate Expression Features

In [3]:
#gene expression features
expression_features = pd.DataFrame({
    "mean_expression": expression.mean(axis=1),
    "expression_variability": expression.std(axis=1)
})

print("Feature table shape:", expression_features.shape)
expression_features.head()

Feature table shape: (20530, 2)


,mean_expression,expression_variability
sample,,
ARHGEF10L,9.538420,0.765509
HIF3A,7.109322,2.222825
RNF17,0.546533,0.938638
RNF10,11.766623,0.344360
RNF11,10.959373,0.492240


In [4]:
output_path = folder / "Datasets for Checkpoint 1" / "LUAD_expression_features.csv"

expression_features.to_csv(output_path)

print("Saved to:", output_path)

Saved to: /Users/skylerskinner/Desktop/Fall 2026 Classes/Datasets for Checkpoint 1/LUAD_expression_features.csv


## Locate the GISTIC files and calculate Amplification Frequencies

In [6]:
from pathlib import Path

data_folder = Path.home() / "Desktop" / "Fall 2026 Classes" / "Datasets for Checkpoint 1"

print(data_folder.exists())

True


In [7]:
for file in data_folder.rglob("*"):
    if file.is_file():
        print(file.name)

.DS_Store
luad_tcga.tar.gz
CRISPRGeneEffect.csv
LUAD_expression_features.csv
subcellular_location.tsv
TCGA.LUAD.sampleMap_HiSeqV2.gz
Model.csv
gene_splits.json
data_mrna_agilent_microarray.txt
meta_mrna_agilent_microarray.txt
meta_cna.txt
meta_clinical_sample.txt
data_mrna_agilent_microarray_zscores_ref_all_samples.txt
meta_rppa.txt
data_linear_cna.txt
data_cna.txt
.DS_Store
LICENSE
data_clinical_patient.txt
meta_mutsig.txt
meta_gistic_genes_del.txt
meta_methylation_hm27.txt
meta_mrna_seq_v2_rsem.txt
data_cna_hg19.seg
data_mrna_agilent_microarray_zscores_ref_diploid_samples.txt
data_mrna_seq_v2_rsem_zscores_ref_diploid_samples.txt
data_clinical_sample.txt
data_rppa_zscores.txt
meta_gistic_genes_amp.txt
data_mrna_seq_v2_rsem.txt
data_methylation_hm27.txt
data_gistic_genes_del.txt
meta_cna_hg19_seg.txt
meta_rppa_zscores.txt
meta_mrna_seq_v2_rsem_zscores_ref_all_samples.txt
data_methylation_hm450.txt
data_rppa.txt
meta_mrna_seq_v2_rsem_zscores_ref_diploid_samples.txt
data_mutsig.txt
data_

In [8]:
import tarfile

archive_path = data_folder / "luad_tcga.tar.gz"

with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(path=data_folder, filter="data")

print("Extraction complete!")

Extraction complete!


In [9]:
cna_path = data_folder / "luad_tcga" / "data_cna.txt"
cna = pd.read_csv(cna_path, sep="\t")

print("Dataset shape:", cna.shape)
print("Columns:", cna.columns[:8].tolist())

cna.iloc[:5, :8]

Dataset shape: (24776, 518)
Columns: ['Hugo_Symbol', 'Entrez_Gene_Id', 'TCGA-05-4244-01', 'TCGA-05-4249-01', 'TCGA-05-4250-01', 'TCGA-05-4382-01', 'TCGA-05-4384-01', 'TCGA-05-4389-01']


,Hugo_Symbol,Entrez_Gene_Id,TCGA-05-4244-01,TCGA-05-4249-01,TCGA-05-4250-01,TCGA-05-4382-01,TCGA-05-4384-01,TCGA-05-4389-01
0,ACAP3,116983.0,-1,-1,-1,0,0,-1
1,ACTRT2,140625.0,-1,-1,-1,0,0,-1
2,AGRN,375790.0,-1,-1,-1,0,0,-1
3,ANKRD65,441869.0,-1,-1,-1,0,0,-1
4,ATAD3A,55210.0,-1,-1,-1,0,0,-1


In [10]:
sample_columns = cna.columns[2:]

cna_values = cna[sample_columns].apply(
    pd.to_numeric, errors="coerce"
)

amplication_features = pd.DataFrame({
    "Gene": cna["Hugo_Symbol"],
    "amplification_frequency": (
        cna_values.eq(2).sum(axis=1)
        / cna_values.notna().sum(axis=1)
    )
})

amplification_features = amplication_features.set_index("Gene")
print("Amplification Features Shape:", amplification_features.shape)
amplification_features.head()

Amplification Features Shape: (24776, 1)


,amplification_frequency
Gene,
ACAP3,0.017442
ACTRT2,0.017442
AGRN,0.017442
ANKRD65,0.017442
ATAD3A,0.017442


## Part 4: Check Gene Names / Resolve Amplification Duplicates and Merge Features

In [11]:
#check for duplicate gene names
print("Expression duplicates:", expression_features.index.duplicated().sum())
print("Amplification duplicates:", amplification_features.index.duplicated().sum())

Expression duplicates: 0
Amplification duplicates: 11


In [12]:
#resolve amplification duplicates
duplicates = amplification_features[
    amplification_features.index.duplicated(keep=False)
]

print(duplicates.sort_index())

            amplification_frequency
Gene                               
ABHD16B                    0.060078
ABHD16B                    0.060078
C1orf220                   0.062016
C1orf220                   0.062016
COMMD9                     0.013566
COMMD9                     0.013566
FOXD4L3                    0.005814
FOXD4L3                    0.005814
GADL1                      0.007752
GADL1                      0.000000
PALM2AKAP2                 0.003876
PALM2AKAP2                 0.003876
PDPK1                      0.005814
PDPK1                      0.005814
SLCO1B3                    0.042636
SLCO1B3                    0.042636
TIMP2                      0.031008
TIMP2                      0.031008
TRAPPC2                    0.005814
TRAPPC2                    0.003876
XAGE1A                     0.007752
XAGE1A                     0.007752


In [13]:
#duplicate genes with different amplification frequencies
conflicting = amplification_features["amplification_frequency"].groupby(level=0).nunique()

conflicting_genes = conflicting[conflicting > 1].index

print("Number of conflicting genes:", len(conflicting_genes))
print("Conflicting genes:", conflicting_genes.tolist())

#remove conflicting genes
amplification_clean = amplification_features.loc[
    ~amplification_features.index.isin(conflicting_genes)
]

#remove identical duplicate entries
amplification_clean = amplification_clean.loc[
    ~amplification_clean.index.duplicated(keep="first")
]

print("Cleaned Shape:", amplification_clean.shape)
print("Remaining duplicates:", amplification_clean.index.duplicated().sum())

Number of conflicting genes: 2
Conflicting genes: ['GADL1', 'TRAPPC2']
Cleaned Shape: (24763, 1)
Remaining duplicates: 0


In [14]:
#merge expression with amplification features
gene_features = expression_features.join(
    amplification_clean,
    how="inner"
)

print("Combined dataset shape:", gene_features.shape)
print("Duplicate genes:", gene_features.index.duplicated().sum())

gene_features.head()

Combined dataset shape: (17926, 3)
Duplicate genes: 0


,mean_expression,expression_variability,amplification_frequency
sample,,,
ARHGEF10L,9.538420,0.765509,0.009690
HIF3A,7.109322,2.222825,0.011628
RNF17,0.546533,0.938638,0.000000
RNF10,11.766623,0.344360,0.001938
RNF11,10.959373,0.492240,0.011628


## Part 5: Load DepMap Data and Identify LUAD Cell Lines

In [15]:
#add depmap Chronos target labels (y)
depmap_path = data_folder / "CRISPRGeneEffect.csv"

depmap = pd.read_csv(depmap_path)

print("Dataset shape:", depmap.shape)
print("First 10 columns:", depmap.columns[:10].tolist())

depmap.head()

Dataset shape: (1361, 18435)
First 10 columns: ['Unnamed: 0', 'A1BG (1)', 'A1CF (29974)', 'A2M (2)', 'A2ML1 (144568)', 'A3GALT2 (127550)', 'A4GALT (53947)', 'A4GNT (51146)', 'AAAS (8086)', 'AACS (65985)']


,Unnamed: 0,A1BG (1),A1CF (29974),A2M (2),A2ML1 (144568),A3GALT2 (127550),A4GALT (53947),A4GNT (51146),AAAS (8086),AACS (65985),...,ZNF43 (7594),ZNF479 (90827),ZNF492 (57615),ZNF587 (84914),ZNF625 (90589),ZNF705B (100132396),ZNF705G (100131980),ZNF98 (148198),ZNG1A (55871),ZNG1B (150472)
0,ACH-002730,0.054161,-0.017049,-0.073392,0.112515,0.029253,-0.063848,0.217973,-0.144079,0.138700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ACH-002731,-0.016594,0.099280,-0.250979,0.166924,-0.463860,0.163275,0.131536,-0.561577,0.020163,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ACH-002736,0.065391,0.274303,0.052888,0.090785,-0.031534,-0.008747,0.056382,-0.319523,0.167751,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ACH-002742,0.002508,-0.079789,0.125623,0.386663,-0.059807,-0.163195,0.223320,-0.257440,0.050843,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ACH-002743,0.048721,0.334458,-0.261417,0.262484,0.142113,0.132199,-0.006062,-0.459170,0.287289,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
#look for LUAD cell lines
for file in data_folder.iterdir():
    print(file.name)

luad_tcga
.DS_Store
luad_tcga.tar.gz
CRISPRGeneEffect.csv
LUAD_expression_features.csv
subcellular_location.tsv
TCGA.LUAD.sampleMap_HiSeqV2.gz
Model.csv
gene_splits.json


In [17]:
model_path = data_folder / "Model.csv"

models = pd.read_csv(model_path)

print("Dataset shape:", models.shape)
print("Columns:", models.columns.tolist())

models.head()

Dataset shape: (2554, 48)
Columns: ['ModelID', 'PatientID', 'CellLineName', 'StrippedCellLineName', 'DepmapModelType', 'OncotreeLineage', 'OncotreePrimaryDisease', 'OncotreeSubtype', 'OncotreeCode', 'RRID', 'Age', 'AgeCategory', 'Sex', 'PatientRace', 'PrimaryOrMetastasis', 'SourceType', 'SourceDetail', 'CatalogNumber', 'SampleCollectionSite', 'ModelSubtypeFeatures', 'PatientSubtypeFeatures', 'TissueOrigin', 'ModelDerivationMaterial', 'ModelTreatment', 'PatientTreatmentStatus', 'PatientTreatmentType', 'PatientTreatmentDetails', 'Stage', 'StagingSystem', 'PatientTumorGrade', 'PatientTreatmentResponse', 'GrowthPattern', 'OnboardedMedia', 'FormulationID', 'SerumFreeMedia', 'PlateCoating', 'EngineeredModel', 'EngineeredModelDetails', 'CulturedResistanceDrug', 'PublicComments', 'CCLEName', 'ModelIDAlias', 'HCMIID', 'ModelType', 'PediatricModelType', 'SangerModelID', 'WTSIMasterCellID', 'COSMICID']


,ModelID,PatientID,CellLineName,StrippedCellLineName,DepmapModelType,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,OncotreeCode,RRID,...,CulturedResistanceDrug,PublicComments,CCLEName,ModelIDAlias,HCMIID,ModelType,PediatricModelType,SangerModelID,WTSIMasterCellID,COSMICID
0,ACH-000001,PT-gj46wT,NIH:OVCAR-3,NIHOVCAR3,HGSOC,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,HGSOC,CVCL_0465,...,NaN,NaN,NIHOVCAR3_OVARY,NaN,NaN,Cell Line,False,SIDM00105,2201.0,905933.0
1,ACH-000002,PT-5qa3uk,HL-60,HL60,AMLMRC,Myeloid,Acute Myeloid Leukemia,AML with Myelodysplasia-Related Changes,AMLMRC,CVCL_0002,...,NaN,NaN,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,NaN,Cell Line,True,SIDM00829,55.0,905938.0
2,ACH-000003,PT-puKIyc,CACO2,CACO2,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,CVCL_0025,...,NaN,NaN,CACO2_LARGE_INTESTINE,NaN,NaN,Cell Line,False,SIDM00891,NaN,NaN
3,ACH-000004,PT-q4K2cp,HEL,HEL,AMLNOS,Myeloid,Acute Myeloid Leukemia,"AML, NOS",AMLNOS,CVCL_0001,...,NaN,NaN,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,NaN,Cell Line,True,SIDM00594,783.0,907053.0
4,ACH-000005,PT-q4K2cp,HEL 92.1.7,HEL9217,AML,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,AML,CVCL_2481,...,NaN,NaN,HEL9217_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,NaN,Cell Line,True,SIDM00593,NaN,NaN


In [18]:
# Identify LUAD cell lines
luad_models = models[models["OncotreeCode"] == "LUAD"].copy()

print("LUAD cell lines in metadata:", len(luad_models))

print(luad_models[
    ["ModelID", "CellLineName", "OncotreeCode"]
].head(10))

LUAD cell lines in metadata: 98
        ModelID CellLineName OncotreeCode
10   ACH-000012      HCC-827         LUAD
19   ACH-000021    NCI-H1693         LUAD
27   ACH-000029  HCC-827-GR5         LUAD
28   ACH-000030        PC-14         LUAD
31   ACH-000033    NCI-H1819         LUAD
60   ACH-000062   RERF-LC-MS         LUAD
64   ACH-000066      HCC4006         LUAD
107  ACH-000109    NCI-H3255         LUAD
119  ACH-000121    NCI-H2405         LUAD
148  ACH-000150      HCC2935         LUAD


In [19]:
#Match LUAD cell lines to DepMap Chronos Data
luad_ids = luad_models["ModelID"].tolist()

luad_depmap = depmap[
    depmap["Unnamed: 0"].isin(luad_ids)
].copy()

print("LUAD cell lines in metadata:", len(luad_ids))
print("LUAD cell lines with Chronos data:", len(luad_depmap))

#display matching cell line IDs
print(luad_depmap["Unnamed: 0"].tolist())

LUAD cell lines in metadata: 98
LUAD cell lines with Chronos data: 53
['ACH-000448', 'ACH-000718', 'ACH-000837', 'ACH-000845', 'ACH-001233', 'ACH-000161', 'ACH-000562', 'ACH-000666', 'ACH-000872', 'ACH-000030', 'ACH-000264', 'ACH-000496', 'ACH-000021', 'ACH-000787', 'ACH-000528', 'ACH-000677', 'ACH-000916', 'ACH-000945', 'ACH-002531', 'ACH-000029', 'ACH-000681', 'ACH-000012', 'ACH-000589', 'ACH-000638', 'ACH-000766', 'ACH-000841', 'ACH-000311', 'ACH-000774', 'ACH-000791', 'ACH-000337', 'ACH-000343', 'ACH-000757', 'ACH-000785', 'ACH-002526', 'ACH-000414', 'ACH-000521', 'ACH-000861', 'ACH-000886', 'ACH-000900', 'ACH-003012', 'ACH-000587', 'ACH-000781', 'ACH-002522', 'ACH-000339', 'ACH-000416', 'ACH-000706', 'ACH-000851', 'ACH-000860', 'ACH-000869', 'ACH-000667', 'ACH-000888', 'ACH-001137', 'ACH-002650']


## Part 6: Calculate Negative Median Chronos Effect for Three LUAD Cell Lines and Merge Labels with Gene Features

In [20]:
#Calculate target labels (y)
#ŷ = -median(Chronos effects across LUAD cell lines)
#each gene needs data from 3 LUAD cell lines

#extract gene effect scores 
luad_effects = luad_depmap.drop(columns=["Unnamed: 0"])

#available LUAD measurements for each gene
valid_counts = luad_effects.notna().sum(axis=0)

#Calculations for y
gene_dependency = -luad_effects.median(axis=0)

#keep genes with 3 LUAD cell lines
gene_dependency = gene_dependency[valid_counts >= 3]

#to dataframe
dependency_labels = gene_dependency.to_frame(
    name="dependency_score"
)

dependency_labels.index.name = "Gene"

print("Genes with valid target labels:", len(dependency_labels))
print("Duplicate gene symbols:", dependency_labels.index.duplicated().sum())

dependency_labels.head()

Genes with valid target labels: 17930
Duplicate gene symbols: 0


,dependency_score
Gene,
A1BG (1),-0.026153
A1CF (29974),0.021362
A2M (2),0.004508
A2ML1 (144568),-0.095499
A3GALT2 (127550),0.024147


In [21]:
#remove DepMap gene IDs in parantheses
dependency_labels.index = dependency_labels.index.str.replace(
    r"\s*\(\d+\)$", "", regex=True
)

print(dependency_labels.head())
print("Duplicate genes:", dependency_labels.index.duplicated().sum())

         dependency_score
Gene                     
A1BG            -0.026153
A1CF             0.021362
A2M              0.004508
A2ML1           -0.095499
A3GALT2          0.024147
Duplicate genes: 0


In [22]:
#merge features with depMap labels
ml_dataset = gene_features.join(
    dependency_labels,
    how="inner"
)

print("Final dataset shape:", ml_dataset.shape)
print("Duplicate genes:", ml_dataset.index.duplicated().sum())
print("Missing values:", ml_dataset.isna().sum())

ml_dataset.head()

Final dataset shape: (15587, 4)
Duplicate genes: 0
Missing values: mean_expression            0
expression_variability     0
amplification_frequency    0
dependency_score           0
dtype: int64


,mean_expression,expression_variability,amplification_frequency,dependency_score
sample,,,,
ARHGEF10L,9.538420,0.765509,0.009690,-0.039245
HIF3A,7.109322,2.222825,0.011628,-0.130178
RNF17,0.546533,0.938638,0.000000,0.086548
RNF10,11.766623,0.344360,0.001938,-0.009211
RNF11,10.959373,0.492240,0.011628,0.195686


## Part 7: Identify Surface-Eligible Genes

In [23]:
#Choose which 2000 surface proteins will be in included in the dataset for a 70/15/15 split
#need to determine how to identify surface-eligible genes
hpa_path = data_folder / "subcellular_location.tsv"
hpa = pd.read_csv(hpa_path, sep="\t")

print("HPA dataset shape:", hpa.shape)
print("Columns:", hpa.columns.tolist())

hpa.head()

HPA dataset shape: (13603, 14)
Columns: ['Gene', 'Gene name', 'Reliability', 'Main location', 'Additional location', 'Extracellular location', 'Enhanced', 'Supported', 'Approved', 'Uncertain', 'Single-cell variation intensity', 'Single-cell variation spatial', 'Cell cycle dependency', 'GO id']


,Gene,Gene name,Reliability,Main location,Additional location,Extracellular location,Enhanced,Supported,Approved,Uncertain,Single-cell variation intensity,Single-cell variation spatial,Cell cycle dependency,GO id
0,ENSG00000000003,TSPAN6,Approved,Cell Junctions;Cytosol,Nucleoli fibrillar center,NaN,NaN,NaN,Cell Junctions;Cytosol;Nucleoli fibrillar center,NaN,Cytosol,NaN,NaN,Cell Junctions (GO:0030054);Cytosol (GO:000582...
1,ENSG00000000457,SCYL3,Supported,Cytosol;Golgi apparatus,NaN,NaN,NaN,Cytosol;Golgi apparatus,NaN,NaN,NaN,NaN,NaN,Cytosol (GO:0005829);Golgi apparatus (GO:0005794)
2,ENSG00000000460,C1orf112,Supported,Nucleoplasm,Nucleoli,NaN,NaN,Nucleoli;Nucleoplasm,NaN,NaN,NaN,NaN,NaN,Nucleoli (GO:0005730);Nucleoplasm (GO:0005654)
3,ENSG00000000938,FGR,Approved,Plasma membrane,Aggresome,NaN,NaN,NaN,Aggresome;Plasma membrane,NaN,NaN,NaN,NaN,Aggresome (GO:0016235);Plasma membrane (GO:000...
4,ENSG00000000971,CFH,Approved,Vesicles,NaN,Predicted to be secreted,NaN,NaN,Vesicles,NaN,NaN,NaN,NaN,Vesicles (GO:0043231)


In [24]:
#use plasma membrane localization
surface_mask = (
    hpa["Main location"].fillna("").str.contains(
        "Plasma membrane", case=False, regex=False
    )
    |
    hpa["Additional location"].fillna("").str.contains(
        "Plasma membrane", case=False, regex=False
    )
)

surface_genes = set(
    hpa.loc[surface_mask, "Gene name"].dropna()
)

ml_dataset["is_surface"] = ml_dataset.index.isin(surface_genes)

print("Surface genes in HPA:", len(surface_genes))
print("Surface genes in ML dataset:", ml_dataset["is_surface"].sum())
print("Non-surface genes:", (~ml_dataset["is_surface"]).sum())

ml_dataset.head()

Surface genes in HPA: 2263
Surface genes in ML dataset: 1955
Non-surface genes: 13632


,mean_expression,expression_variability,amplification_frequency,dependency_score,is_surface
sample,,,,,
ARHGEF10L,9.538420,0.765509,0.009690,-0.039245,False
HIF3A,7.109322,2.222825,0.011628,-0.130178,True
RNF17,0.546533,0.938638,0.000000,0.086548,False
RNF10,11.766623,0.344360,0.001938,-0.009211,False
RNF11,10.959373,0.492240,0.011628,0.195686,False


## Part 8: Select 2000 Genes and Create the 70/15/15 Split

In [25]:
#Select 2000 genes
from sklearn.model_selection import train_test_split

surface = ml_dataset[ml_dataset["is_surface"]]
non_surface = ml_dataset[~ml_dataset["is_surface"]]

#12.5% of the dataset is surface-eligible
surface_sample = surface.sample(n=250, random_state=42)
non_surface_sample = non_surface.sample(n=1750, random_state=42)

selected_genes = pd.concat([surface_sample, non_surface_sample])

selected_genes = selected_genes.sample(frac=1, random_state=42)

print("Selected genes:", len(selected_genes))
print("Surface genes:", selected_genes["is_surface"].sum())
print("Non-surface genes:", (~selected_genes["is_surface"]).sum())
print("Duplicate genes:", selected_genes.index.duplicated().sum())

Selected genes: 2000
Surface genes: 250
Non-surface genes: 1750
Duplicate genes: 0


In [26]:
#Create the 70/15/15 splits (70% training, 15% validation, 15% testing)
train, temp = train_test_split(
    selected_genes,
    test_size=0.30,
    stratify=selected_genes["is_surface"],
    random_state=42
)

val, test = train_test_split(
    temp,
    test_size=0.50,
    stratify=temp["is_surface"],
    random_state=42
)

#check split sizes
for name, df in [
    ("Training", train),
    ("Validation", val),
    ("Testing", test)
]:
    print(
        name,
        "genes", len(df),
        "| Surface genes:", df["is_surface"].sum()
    )

#check that a gene does not appear in other splits
print("Train/val overlap:", len(train.index.intersection(val.index)))
print("Train/test overlap:", len(train.index.intersection(test.index)))
print("Val/test overlap:", len(val.index.intersection(test.index)))

Training genes 1400 | Surface genes: 175
Validation genes 300 | Surface genes: 38
Testing genes 300 | Surface genes: 37
Train/val overlap: 0
Train/test overlap: 0
Val/test overlap: 0


## Part 9: Save Split Definitions and Display the Table

In [27]:
#save the split implementation
import json

split_dict = {
    "train": train.index.tolist(),
    "validation": val.index.tolist(),
    "test": test.index.tolist()
}

with open(data_folder / "gene_splits.json", "w") as f:
    json.dump(split_dict, f, indent=2)

print("Saved gene splits!")

Saved gene splits!


In [28]:
#Display a table showing the three input features, target label, and surface eligibility for the 2000 genes
display(selected_genes[
    [
        "mean_expression",
        "expression_variability",
        "amplification_frequency",
        "dependency_score",
        "is_surface"
    ]
].head(20))

,mean_expression,expression_variability,amplification_frequency,dependency_score,is_surface
sample,,,,,
NSUN5,9.334394,0.658270,0.005814,0.164239,False
CUX2,1.856583,1.096357,0.003876,0.123804,False
ARSH,1.507263,1.285854,0.000000,-0.089057,False
PPCDC,7.556817,0.595786,0.005814,0.181608,False
OPALIN,0.049782,0.205171,0.001938,-0.050147,False
COL22A1,4.707653,2.103355,0.062016,-0.123931,False
ASTN1,2.066297,1.845089,0.062016,-0.012893,False
FAR2,6.153266,1.381342,0.036822,0.145091,False
ERAP1,10.634211,0.738716,0.001938,0.063324,True


## Part 10: Enrichment Data Findings and Calculate Enrichment

In [29]:
#find existing dataframes
for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        print(name, value.shape)

___ (5, 5)
expression (20530, 576)
expression_features (20530, 2)
_3 (5, 2)
cna (24776, 518)
_9 (5, 8)
cna_values (24776, 516)
amplication_features (24776, 2)
amplification_features (24776, 1)
duplicates (22, 1)
amplification_clean (24763, 1)
gene_features (17926, 3)
depmap (1361, 18435)
_15 (5, 18435)
models (2554, 48)
_17 (5, 48)
luad_models (98, 48)
luad_depmap (53, 18435)
luad_effects (53, 18434)
dependency_labels (17930, 1)
_20 (5, 1)
ml_dataset (15587, 5)
hpa (13603, 14)
_23 (5, 14)
surface (1955, 5)
non_surface (13632, 5)
surface_sample (250, 5)
non_surface_sample (1750, 5)
selected_genes (2000, 5)
train (1400, 5)
temp (600, 5)
val (300, 5)
test (300, 5)
df (300, 5)


In [30]:
print("Expression dataset shape:", expression.shape)
print("First 10 columns:", expression.columns[:10].tolist())
print("First 5 gene names:", expression.index[:5].tolist())

display(expression.iloc[:5, :5])

Expression dataset shape: (20530, 576)
First 10 columns: ['TCGA-69-7978-01', 'TCGA-62-8399-01', 'TCGA-78-7539-01', 'TCGA-50-5931-11', 'TCGA-73-4658-01', 'TCGA-44-6775-01', 'TCGA-44-2655-01', 'TCGA-44-3398-01', 'TCGA-62-8397-01', 'TCGA-53-A4EZ-01']
First 5 gene names: ['ARHGEF10L', 'HIF3A', 'RNF17', 'RNF10', 'RNF11']


,TCGA-69-7978-01,TCGA-62-8399-01,TCGA-78-7539-01,TCGA-50-5931-11,TCGA-73-4658-01
sample,,,,,
ARHGEF10L,9.9898,10.4257,9.6264,8.6835,9.2078
HIF3A,4.2598,11.6239,9.1362,9.4824,5.0288
RNF17,0.4181,0.0000,1.1231,0.8221,0.0000
RNF10,10.3657,11.5489,11.6692,11.7341,11.6209
RNF11,11.1718,11.0200,10.4679,11.6787,11.3414


In [31]:
#tumor and normal samples
tumor_cols = [
    col for col in expression.columns
    if str(col).split("-")[-1][:2] == "01"
]

normal_cols = [
    col for col in expression.columns
    if str(col).split("-")[-1][:2] == "11"
]

print("Tumor samples:", len(tumor_cols))
print("Normal samples:", len(normal_cols))
print("Total samples:", expression.shape[1])

Tumor samples: 515
Normal samples: 59
Total samples: 576


In [32]:
#Calculate enrichment
tumor_mean = expression[tumor_cols].mean(axis=1)
normal_mean = expression[normal_cols].mean(axis=1)

enrichment = (tumor_mean - normal_mean).rename("enrichment")

print("Genes with enrichment values:", enrichment.notna().sum())
display(enrichment.head(10))

Genes with enrichment values: 20530


sample
ARHGEF10L    0.022851
HIF3A       -2.164437
RNF17        0.080450
RNF10        0.031455
RNF11       -0.611914
RNF13       -0.813429
GTF2IP1      0.126136
REM1        -1.788120
MTVR2       -0.221997
RTN4RL2      2.172268
Name: enrichment, dtype: float64

## Part 11: Reformat the Checkpoint 1 Table

In [33]:
#Place enrichment into the cell
#create final checkpoint 1 table
checkpoint_table = selected_genes.rename(columns={
    "mean_expression": "Expression",
    "enrichment": "Enrichment",
    "expression_variability": "Variability",
    "amplification_frequency": "Amplification",
    "dependency_score": "Dependency Score",
    "is_surface": "Surface Level"
}).copy()

checkpoint_table["Enrichment"] = enrichment.reindex(checkpoint_table.index)

checkpoint_table["Surface Level"] = checkpoint_table["Surface Level"].map({
    True: "Yes",
    False: "No"
})

#rearrange columns
checkpoint_table = checkpoint_table[
    [
        "Expression",
        "Enrichment",
        "Variability",
        "Amplification",
        "Dependency Score",
        "Surface Level"
    ]
]

checkpoint_table.index.name = "Gene"

print("Dataset shape:", checkpoint_table.shape)
print("Missing values:")
print(checkpoint_table.isna().sum())

display(checkpoint_table.head(20))

Dataset shape: (2000, 6)
Missing values:
Expression          0
Enrichment          0
Variability         0
Amplification       0
Dependency Score    0
Surface Level       0
dtype: int64


,Expression,Enrichment,Variability,Amplification,Dependency Score,Surface Level
Gene,,,,,,
NSUN5,9.334394,0.939807,0.658270,0.005814,0.164239,No
CUX2,1.856583,0.112388,1.096357,0.003876,0.123804,No
ARSH,1.507263,1.586691,1.285854,0.000000,-0.089057,No
PPCDC,7.556817,0.218856,0.595786,0.005814,0.181608,No
OPALIN,0.049782,0.042464,0.205171,0.001938,-0.050147,No
COL22A1,4.707653,2.505478,2.103355,0.062016,-0.123931,No
ASTN1,2.066297,-1.337623,1.845089,0.062016,-0.012893,No
FAR2,6.153266,-1.845173,1.381342,0.036822,0.145091,No
ERAP1,10.634211,-0.350305,0.738716,0.001938,0.063324,Yes


## Part 12: Verify Splits and Input Features

In [34]:
#add enrichment to existing splits
for split in [train, val, test]:
    split["enrichment"] = enrichment.reindex(split.index)

#check split sizes and missing values
for name, split in [("Training", train),
                    ("Validation", val),
                    ("Testing", test)]:
    print(f"{name}: {len(split)} genes")
    print("Missing enrichment:", split["enrichment"].isna().sum())
    print()

Training: 1400 genes
Missing enrichment: 0

Validation: 300 genes
Missing enrichment: 0

Testing: 300 genes
Missing enrichment: 0



In [35]:
#verify the four input features
feature_cols = [
    "mean_expression",
    "enrichment",
    "expression_variability",
    "amplification_frequency"
]

#training input and target
X_train = train[feature_cols]
y_train = train["dependency_score"]

#validation input and target
X_val = val[feature_cols]
y_val = val["dependency_score"]

#testing input and target
X_test = test[feature_cols]
y_test = test["dependency_score"]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

display(X_train.head())

Training: (1400, 4) (1400,)
Validation: (300, 4) (300,)
Testing: (300, 4) (300,)


,mean_expression,enrichment,expression_variability,amplification_frequency
sample,,,,
PRSS16,8.752109,0.618184,0.862797,0.015504
DBNDD1,9.049138,1.864264,1.091127,0.003876
TAF2,9.412092,0.618644,0.554447,0.067829
C10orf120,0.017178,0.019213,0.117435,0.001938
ATG2A,9.858161,0.238508,0.568847,0.003876


In [36]:
#check for missing values
for name, X, y in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val),
    ("Testing", X_test, y_test)
]:
    print(name)
    print("Missing input values:", X.isna().sum().sum())
    print("Missing target values:", y.isna().sum())

Training
Missing input values: 0
Missing target values: 0
Validation
Missing input values: 0
Missing target values: 0
Testing
Missing input values: 0
Missing target values: 0


## Part 13: Save and Export Checkpoint 1 Data 

In [37]:
#Save the prepared dataset and splits for Checkpoint 2
checkpoint_table.to_csv("checkpoint1_gene_dataset.csv")

train.to_csv("checkpoint1_train.csv")
val.to_csv("checkpoint1_validation.csv")
test.to_csv("checkpoint1_test.csv")

print("Checkpoint 1 datasets saved!")

Checkpoint 1 datasets saved!
